# F1-scientific-python — Practice p22

**Type:** constrained coding · **Difficulty:** intro · **Concepts:** seaborn-programming

**Set A — fundamentals · Budget: 25 minutes.**

Implement exactly `def plot_score_hist(scores):`. `scores` is a 1-D NumPy array of integer quiz scores from 0 through 100.

Your function must:

- create an explicit `fig, ax` with `plt.subplots(figsize=(6, 4))`;
- call `sns.histplot` exactly once with the array passed through `x=`, the fixed edge array `np.arange(-0.5, 110.0, 10.0)` passed through `bins=`, and `ax=ax`;
- use `color="steelblue"` and `edgecolor="black"`;
- set the exact title `Quiz-score distribution`, x-label `score`, y-label `count`, and x-limits `(-0.5, 109.5)`;
- return `(fig, ax)` and not call `plt.show()` inside the function.

**Banned (zero points): `plt.hist` inside `plot_score_hist`, and any input container other than the supplied NumPy array.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgba

In [ ]:
def plot_score_hist(scores):
    # YOUR CODE HERE
    ...

Run the deterministic contract check below. It checks the axes and bar counts rather than comparing pixels. Do not edit the check.

In [ ]:
probe_scores = np.array([0, 4, 10, 19, 20, 39, 40, 58, 60, 79, 80, 99, 100])
histplot_calls = []
_original_histplot = sns.histplot
_original_show = plt.show

def _record_histplot(*args, **kwargs):
    histplot_calls.append((args, kwargs.copy()))
    return _original_histplot(*args, **kwargs)

def _forbid_show(*args, **kwargs):
    raise AssertionError("plot_score_hist must not call plt.show()")

sns.histplot = _record_histplot
plt.show = _forbid_show
try:
    fig, ax = plot_score_hist(probe_scores)
finally:
    sns.histplot = _original_histplot
    plt.show = _original_show

edges = np.arange(-0.5, 110.0, 10.0)
expected_counts, _ = np.histogram(probe_scores, bins=edges)
patches = list(ax.patches)
drawn_counts = np.array([patch.get_height() for patch in patches], dtype=int)

assert len(histplot_calls) == 1
call_args, call_kwargs = histplot_calls[0]
assert call_args == ()
required_kwargs = {"x", "bins", "ax", "color", "edgecolor"}
assert required_kwargs <= set(call_kwargs)
assert call_kwargs["x"] is probe_scores
assert np.array_equal(call_kwargs["bins"], edges)
assert call_kwargs["ax"] is ax
assert call_kwargs["color"] == "steelblue"
assert call_kwargs["edgecolor"] == "black"
assert len(patches) == len(edges) - 1
assert all(patch.get_visible() for patch in patches)
assert all((patch.get_alpha() if patch.get_alpha() is not None
            else patch.get_facecolor()[3]) > 0 for patch in patches)
assert np.array_equal(drawn_counts, expected_counts)
assert drawn_counts.sum() == len(probe_scores)
assert np.allclose([patch.get_x() for patch in patches], edges[:-1], atol=1e-12, rtol=0)
assert np.allclose([patch.get_width() for patch in patches], np.diff(edges), atol=1e-12, rtol=0)
assert all(np.allclose(patch.get_facecolor()[:3], to_rgba("steelblue")[:3], atol=1e-12, rtol=0) for patch in patches)
assert all(np.allclose(patch.get_edgecolor(), to_rgba("black"), atol=1e-12, rtol=0) for patch in patches)
assert np.allclose(fig.get_size_inches(), (6, 4), atol=1e-12, rtol=0)
assert ax.get_title() == "Quiz-score distribution"
assert ax.get_xlabel() == "score" and ax.get_ylabel() == "count"
assert np.allclose(ax.get_xlim(), (-0.5, 109.5), atol=1e-12, rtol=0)
plt.close(fig)